In [2]:
import pandas as pd

train_df = pd.read_csv('final_proj_data.csv')

# 1. Оцінка розміру та пропусків
total_rows, total_cols = train_df.shape
total_missing = train_df.isna().sum().sum()

print(f"Розмір датасету: {total_rows} рядків, {total_cols} колонок")
print(f"Загальна кількість пропусків (NaN) у всьому датасеті: {total_missing}")

# 2. Оцінка дисбалансу цільової змінної 'y'
print("\nРозподіл класів (у відсотках):")
print(train_df['y'].value_counts(normalize=True) * 100)

# 3. Перевірка перших 5 рядків для розуміння структури
display(train_df.head())

Розмір датасету: 10000 рядків, 231 колонок
Загальна кількість пропусків (NaN) у всьому датасеті: 1603449

Розподіл класів (у відсотках):
y
0    86.95
1    13.05
Name: proportion, dtype: float64


,Var1,Var2,Var3,Var4,Var5,Var6,Var7,Var8,Var9,Var10,...,Var222,Var223,Var224,Var225,Var226,Var227,Var228,Var229,Var230,y
0,NaN,NaN,NaN,NaN,NaN,812.0,14.0,NaN,NaN,NaN,...,catzS2D,jySVZNlOJy,NaN,xG3x,Aoh3,ZI9m,ib5G6X1eUxUn6,mj86,NaN,0
1,NaN,NaN,NaN,NaN,NaN,2688.0,7.0,NaN,NaN,NaN,...,i06ocsg,LM8l689qOp,NaN,kG3k,WqMG,RAYp,55YFVY9,mj86,NaN,0
2,NaN,NaN,NaN,NaN,NaN,1015.0,14.0,NaN,NaN,NaN,...,P6pu4Vl,LM8l689qOp,NaN,kG3k,Aoh3,ZI9m,R4y5gQQWY8OodqDV,am7c,NaN,0
3,NaN,NaN,NaN,NaN,NaN,168.0,0.0,NaN,NaN,NaN,...,BNrD3Yd,LM8l689qOp,NaN,NaN,FSa2,RAYp,F2FyR07IdsN7I,NaN,NaN,0
4,NaN,NaN,NaN,NaN,NaN,14.0,0.0,NaN,NaN,NaN,...,3B1QowC,LM8l689qOp,NaN,NaN,WqMG,RAYp,F2FyR07IdsN7I,NaN,NaN,0


In [3]:
# 1. Аналіз пропусків на рівні окремих колонок
missing_percent = train_df.isna().mean() * 100

print(f"Колонок, де більше 50% пропусків: {len(missing_percent[missing_percent > 50])}")
print(f"Колонок, де більше 80% пропусків: {len(missing_percent[missing_percent > 80])}")
print(f"Колонок, де більше 95% пропусків: {len(missing_percent[missing_percent > 95])}")

# 2. Аналіз категоріальних ознак (Кардинальність)
# Відбираємо всі колонки з текстом (object)
cat_cols = train_df.select_dtypes(include=['object']).columns
cardinality = train_df[cat_cols].nunique()

print("\nТоп-10 категоріальних колонок за кількістю унікальних значень:")
print(cardinality.sort_values(ascending=False).head(10))

Колонок, де більше 50% пропусків: 159
Колонок, де більше 80% пропусків: 154
Колонок, де більше 95% пропусків: 153

Топ-10 категоріальних колонок за кількістю унікальних значень:
Var217    5529
Var200    4478
Var214    4478
Var202    3802
Var220    2100
Var222    2100
Var198    2100
Var199    1850
Var216     977
Var192     297
dtype: int64


C:\Users\lepik\AppData\Local\Temp\ipykernel_16192\396540565.py:10: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cat_cols = train_df.select_dtypes(include=['object']).columns


In [ ]:
import pandas as pd
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, TargetEncoder
from sklearn.ensemble import HistGradientBoostingClassifier
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

train_df = pd.read_csv('final_proj_data.csv')
test_df = pd.read_csv('final_proj_test.csv')

X = train_df.drop(columns=['y'])
y = train_df['y']


# А. Видаляємо колонки, де більше 80% пропусків
missing_percent = X.isna().mean()
cols_to_drop_na = missing_percent[missing_percent > 0.80].index.tolist()
print(f"Видаляємо через пропуски (>80%): {len(cols_to_drop_na)} колонок")

# Б. Видаляємо текстові колонки з аномальною кардинальністю (> 3000)
cat_cols_initial = X.select_dtypes(include=['object', 'string']).columns
cardinality = X[cat_cols_initial].nunique()
cols_to_drop_cardinality = cardinality[cardinality > 3000].index.tolist()
print(f"Видаляємо через кардинальність (>3000 унікальних): {len(cols_to_drop_cardinality)} колонок")

all_cols_to_drop = list(set(cols_to_drop_na + cols_to_drop_cardinality))
X = X.drop(columns=all_cols_to_drop)
test_df_clean = test_df.drop(columns=all_cols_to_drop)


# Динамічно визначаємо, які колонки залишилися
cat_cols = X.select_dtypes(include=['object', 'string']).columns.tolist()
num_cols = X.select_dtypes(exclude=['object', 'string']).columns.tolist()

# Числа: медіана + масштабування
num_transformer = ImbPipeline(steps=[
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Категорії: заповнюємо спецзначенням 'missing' + TargetEncoder
# TargetEncoder замінює категорію на середню ймовірність класу 1 для цієї категорії
cat_transformer = ImbPipeline(steps=[
    ('imputer', SimpleImputer(strategy='constant', fill_value='missing')),
    ('encoder', TargetEncoder(target_type='binary', smooth='auto'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', num_transformer, num_cols),
        ('cat', cat_transformer, cat_cols)
    ]
)

# Фінальний Pipeline з балансуванням (SMOTE)
pipeline = ImbPipeline(steps=[
    ('preprocessor', preprocessor),
    ('smote', SMOTE(random_state=42)),
    ('classifier', HistGradientBoostingClassifier(random_state=42))
])

# 4. Оцінка (Крос-валідація)
print("\nКрос-валідація.")
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores = cross_val_score(pipeline, X, y, cv=cv, scoring='balanced_accuracy', n_jobs=-1)

print(f"Balanced Accuracy: {scores.mean():.4f} (+/- {scores.std():.4f})")

print("Навчання фінальної моделі на всіх даних...")
pipeline.fit(X, y)
predictions = pipeline.predict(test_df_clean)

submission = pd.DataFrame({
    'index': test_df['index'] if 'index' in test_df.columns else test_df.index,
    'y': predictions
})

submission.to_csv('submission_v2.csv', index=False)

Видаляємо через пропуски (>80%): 154 колонок
Видаляємо через кардинальність (>3000 унікальних): 4 колонок

Проводимо крос-валідацію. Це може зайняти хвилину...
Balanced Accuracy після очищення: 0.8813 (+/- 0.0091)
Навчання фінальної моделі на всіх даних...
Файл submission_v2.csv успішно створено! Можеш відправляти на Kaggle.
